In [1]:
# Setup — run this cell first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 4.5)

print("✅ Libraries loaded — you're ready to go!")

✅ Libraries loaded — you're ready to go!


In [24]:
import json
import requests

def normalize_for_hdb_api(onemap_street_name):
        # 1. Force Uppercase to match data.gov.sg schema
        street = onemap_street_name.upper().strip()

        # 2. Dictionary mapping of OneMap values -> HDB Dataset abbreviations
        abbreviations = {
            " AVENUE": " AVE",
            " ROAD": " RD",
            " DRIVE": " DR",
            " STREET": " ST",
            " BOULEVARD": " BLVD",
            " CRESCENT": " CRES",
            " CLOSE": " CL",
            " CENTRAL": " CTRL",
            " BUKIT": " BT",
            " TANJONG": " TG",
            " KAMPONG": " KG",
            " NORTH": " NTH"
        }

        # Replace words sequentially
        for full_word, short_word in abbreviations.items():
            if full_word in street:
                street = street.replace(full_word, short_word)

        return street
# Let's say your user input or OneMap gave you: "Tampines North Drive 2"
raw_onemap_street = "Tampines North Drive 2"
cleaned_street = normalize_for_hdb_api(raw_onemap_street)

url = "https://data.gov.sg/api/action/datastore_search"

# Correct mapping: Separate block number from street name 
# and ensure exact column IDs match the dataset schema
filter_conditions = {
    "blk_no": "635B", 
    "street": cleaned_street 
}

params = {
    "resource_id": "d_17f5382f26140b1fdae0ba2ef6239d2f",
    "filters": json.dumps(filter_conditions),
    "fields": "year_completed,bldg_contract_town",
    "limit": 5
}

response = requests.get(url, params=params)
print(response.json())

{'success': True, 'result': {'resource_id': 'd_17f5382f26140b1fdae0ba2ef6239d2f', 'fields': [{}, {}], 'records': [{'year_completed': '2022', 'bldg_contract_town': 'TAP'}], '_links': {'start': '/api/action/datastore_search?resource_id=d_17f5382f26140b1fdae0ba2ef6239d2f&limit=5&filters=%7B%22blk_no%22%3A%20%22635B%22%2C%20%22street%22%3A%20%22TAMPINES%20NTH%20DR%202%22%7D&fields=year_completed,bldg_contract_town', 'next': '/api/action/datastore_search?resource_id=d_17f5382f26140b1fdae0ba2ef6239d2f&offset=5&limit=5&filters=%7B%22blk_no%22%3A%20%22635B%22%2C%20%22street%22%3A%20%22TAMPINES%20NTH%20DR%202%22%7D&fields=year_completed,bldg_contract_town'}, 'total': 1, 'limit': 5, 'filters': '{"blk_no": "635B", "street": "TAMPINES NTH DR 2"}'}}


In [2]:
import pandas as pd
import numpy as np

# Load dataset (Replace with your downloaded CSV path)
df = pd.read_parquet("data/hdb_resale_price.parquet")
pd.set_option('display.max_columns', None)

# View your DataFrame
df.head()


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,full_address,lat,lon,x,y,dist_to_jurong_lake_district,dist_to_tampines_regional_centre,dist_to_woodlands_regional_centre,dist_to_paya_lebar_central,dist_to_one_north_hub,dist_to_punggol_digital_district,dist_to_bishan_sub_regional_centre,dist_to_seletar_aerospace_park,dist_to_changi_business_park,dist_to_international_business_park,min_distance_to_regional_hub_km,closest_regional_hub_name,primary_schools_within_1km,primary_schools_within_2km,closest_primary_school_name,dist_to_closest_primary_school_km,mrt_within_500m,mrt_within_1km,closest_mrt_name,dist_to_closest_mrt_km,lrt_within_500m,closest_lrt_name,dist_to_closest_lrt_km,malls_within_500m,malls_within_1km,closest_shopping_mall_name,dist_to_closest_shopping_mall_km
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0,406 ANG MO KIO AVE 10,1.362005,103.853880,30288.234663,38229.067463,12.586295,11.026219,9.902276,6.127222,9.520345,6.567731,1.106705,4.521283,13.860850,11.891477,1.106705,to_bishan_sub_regional_centre,2,10,Townsville Primary School,0.218125,0,2,Ang Mo Kio Mrt Station (Ns16),0.999942,0,Fernvale Lrt Station (Sw5),4.140950,0,0,Bishan North Shopping Mall,1.101454
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0,108 ANG MO KIO AVE 4,1.370966,103.838202,28543.458747,39220.009892,11.196611,12.896038,7.944305,8.096704,9.130480,7.668190,2.267672,4.587861,15.805031,10.561282,2.267672,to_bishan_sub_regional_centre,2,5,Ang Mo Kio Primary School,0.241572,1,1,Mayflower Mrt Station (Te6),0.189980,0,Fernvale Lrt Station (Sw5),4.830643,0,1,Amk Hub,0.650857
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0,602 ANG MO KIO AVE 5,1.380709,103.835368,28228.099954,40297.283149,11.327516,13.431444,6.994327,9.044019,9.875597,7.666120,3.364860,4.151415,16.435721,10.746807,3.364860,to_bishan_sub_regional_centre,2,3,Anderson Primary School,0.777155,0,1,Lentor Mrt Station (Te5),0.532155,0,Fernvale Lrt Station (Sw5),4.720791,0,0,Amk Hub,1.527662
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0,465 ANG MO KIO AVE 10,1.366201,103.857201,30657.824693,38693.098657,13.057117,10.723364,9.895713,6.194440,10.107784,6.003814,1.696592,3.968240,13.627876,12.374500,1.696592,to_bishan_sub_regional_centre,3,10,Teck Ghee Primary School,0.698166,0,2,Ang Mo Kio Mrt Station (Ns16),0.945372,0,Fernvale Lrt Station (Sw5),3.547948,0,0,Amk Hub,1.530310
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0,601 ANG MO KIO AVE 5,1.381041,103.835132,28201.782487,40334.051212,11.319946,13.466058,6.950719,9.088378,9.894647,7.684167,3.409488,4.154318,16.472824,10.741548,3.409488,to_bishan_sub_regional_centre,1,3,Anderson Primary School,0.782553,1,1,Lentor Mrt Station (Te5),0.498419,0,Fernvale Lrt Station (Sw5),4.736754,0,0,Amk Hub,1.572811


In [3]:
# Parse Complex Remaining Lease strings (e.g., '61 years 04 months')
def parse_lease(lease_str):
    if pd.isna(lease_str):
        return 99.0
    parts = str(lease_str).split()
    years = int(parts[0])
    months = int(parts[2]) if 'month' in parts else 0
    return years + (months / 12.0)

df['remaining_lease_years'] = df['remaining_lease'].apply(parse_lease)

# B. Decode Storey Range to Numerical Midpoint (e.g., '10 TO 12' -> 11.0)
df['storey_midpoint'] = df['storey_range'].apply(lambda x: sum(map(int, str(x).split(' TO '))) / 2)

# C. Extract Numeric Time Values from Month (e.g., '2017-01' -> Year 2017, Month 1)
df['year'] = df['month'].apply(lambda x: int(str(x).split('-')[0]))
df['month_num'] = df['month'].apply(lambda x: int(str(x).split('-')[1]))

#df = df[df['year'] >= 2022].reset_index(drop=True)

# --- FEATURE SELECTION & SEPARATION ---
# Select the structural numeric values you generated
features =  [
    'floor_area_sqm', 'lease_commence_date', 'remaining_lease_years', 'storey_midpoint',
    'year', 'month_num', 'min_distance_to_regional_hub_km', 'primary_schools_within_1km', 
    'primary_schools_within_2km', 'dist_to_closest_primary_school_km', 'mrt_within_500m', 
    'mrt_within_1km', 'dist_to_closest_mrt_km', 'lrt_within_500m', 'dist_to_closest_lrt_km', 
    'malls_within_500m', 'malls_within_1km', 'dist_to_closest_shopping_mall_km',
    'town', 'street_name', 'closest_mrt_name', 'closest_primary_school_name', 'flat_model', 'flat_type'
]
X_raw = df[features].copy()
y = df['resale_price']

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error, root_mean_squared_error

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42)

# Create clean working copies for encoding
X_train = X_train_raw.copy()
X_test = X_test_raw.copy()



# --- APPLY TARGET ENCODING ---
target_encode_features = ['street_name', 'closest_mrt_name', 'closest_primary_school_name', 'flat_model', 'flat_type']

for col in target_encode_features:
    # Compute mean price per category using only Training Data
    mean_mapped = y_train.groupby(X_train[col]).mean()
    
    # Map back to train and test sets
    X_train[f'{col}_encoded'] = X_train[col].map(mean_mapped)
    X_test[f'{col}_encoded'] = X_test[col].map(mean_mapped)
    
    # Fill any unseen test categories with global training price average
    X_train[f'{col}_encoded'] = X_train[f'{col}_encoded'].fillna(y_train.mean())
    X_test[f'{col}_encoded'] = X_test[f'{col}_encoded'].fillna(y_train.mean())

# Drop the original text columns used for target encoding
X_train = X_train.drop(columns=target_encode_features)
X_test = X_test.drop(columns=target_encode_features)

# --- ONE-HOT ENCODE TOWN SEPARATELY AND ALIGN ---
# One-hot encode town on train and test independently
X_train = pd.get_dummies(X_train, columns=['town'], dtype=int)
X_test = pd.get_dummies(X_test, columns=['town'], dtype=int)

# It forces X_test to have the exact same columns, in the exact same order as X_train.
# If X_test has a unique town, it gets dropped. If it's missing a town, it gets added with all 0s.
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# --- TRAIN LIGHTGBM ---
# Both X_train and X_test are now guaranteed to have the exact same number of columns!
print(f"Final Training Features Count: {X_train.shape[1]}")
print(f"Final Testing Features Count: {X_test.shape[1]}")

model = LGBMRegressor(n_estimators=1500, learning_rate=0.03, num_leaves=127, random_state=42)
model.fit(X_train, y_train)

# 1. Generate predictions using the aligned test set
predictions = model.predict(X_test)

# 2. Calculate accuracy metrics
mae = mean_absolute_error(y_test, predictions)
mape_baseline = mean_absolute_percentage_error(y_test, predictions)
rsme_baseline = root_mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Mean Absolute Error: ${mae:,.2f}")
print(f"LightGBM MAPE = {mape_baseline:.1%}")
print(f"LightGBM RMSE = ${rsme_baseline:,.2f}")  
print(f"LightGBM R² Score: {r2 * 100:.2f}%")


Final Training Features Count: 49
Final Testing Features Count: 49
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009643 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2375
[LightGBM] [Info] Number of data points in the train set: 186091, number of used features: 49
[LightGBM] [Info] Start training from score 530253.113057
Mean Absolute Error: $18,467.27
LightGBM MAPE = 3.6%
LightGBM RMSE = $25,877.07
LightGBM R² Score: 98.14%


In [29]:
# --- TEST ---
# Input raw values exactly matching your dataset columns
sample_flat_raw = {
    'month': '2026-06',
    'town': 'ANG MO KIO',
    'flat_type': '3 ROOM',
    'storey_range': '10 TO 12',
    'floor_area_sqm': 68.0,
    'lease_commence_date': 1980,
    'remaining_lease_years': 62.08, # Handled raw text conversions previously
    'storey_midpoint': 11.0,
    'year': 2026,
    'month_num': 6,
    'flat_model': 'New Generation',
    'street_name': 'ANG MO KIO AVE 10',
    'closest_mrt_name': 'Ang Mo Kio Mrt Station (Ns16)',
    'closest_primary_school_name': 'Teck Ghee Primary School',
    
    # Advanced proximity metrics matching your dataset schema
    'min_distance_to_regional_hub_km': 1.69,
    'primary_schools_within_1km': 3,
    'primary_schools_within_2km': 10,
    'dist_to_closest_primary_school_km': 0.69,
    'mrt_within_500m': 0,
    'mrt_within_1km': 2,
    'dist_to_closest_mrt_km': 0.94,
    'lrt_within_500m': 0,
    'dist_to_closest_lrt_km': 3.54,
    'malls_within_500m': 0,
    'malls_within_1km': 0,
    'dist_to_closest_shopping_mall_km': 1.53
}

# Convert single dictionary row into a Pandas DataFrame
X_sample = pd.DataFrame([sample_flat_raw])

# --- 2. APPLY TARGET ENCODING MAPS FROM TRAINING ---
target_encode_features = ['street_name', 'closest_mrt_name', 'closest_primary_school_name', 'flat_model', 'flat_type']

for col in target_encode_features:
    # Uses 'mean_mapped' Series generated inside your Step 5 training loop
    X_sample[f'{col}_encoded'] = X_sample[col].map(mean_mapped)
    
    # Fallback to global training mean if the string wasn't in your training data split
    X_sample[f'{col}_encoded'] = X_sample[f'{col}_encoded'].fillna(y_train.mean())

# Drop original unencoded text columns to prevent layout corruption
X_sample = X_sample.drop(columns=target_encode_features)

# --- 3. ONE-HOT ENCODE TOWN & ALIGN FEATURES ---
X_sample_encoded = pd.get_dummies(X_sample, columns=['town'], dtype=int)

# Crucial Guardrail: Forces the sample row to match X_train exactly (Solves 48 vs 49 error)
X_sample_final = X_sample_encoded.reindex(columns=X_train.columns, fill_value=0)

# --- 4. RUN MODEL VALUATION ---
predicted_valuation = model.predict(X_sample_final)

print("\n========================== VALUATION REPORT ==========================")
print(f" Target Features Count Verified: {X_sample_final.shape}")
print(f" Estimated HDB Resale Value: ${predicted_valuation[0]:,.2f}")
print("=======================================================================")



========================== VALUATION REPORT ==========================
 Target Features Count Verified: (1, 49)
 Estimated HDB Resale Value: $450,307.89


In [30]:
import pickle

# Bundle your model artifacts together
model_artifacts = {
    'model': model,                                     # Your trained LightGBM model
    'trained_features': list(X_train.columns),          # Aligned column list
    'global_price_mean': y_train.mean(),                # Global fallback average
    'target_encoding_maps': {                           # All your mean_mapped Series
        col: y_train.groupby(X_raw[col]).mean() for col in [
            'street_name', 'closest_mrt_name', 'closest_primary_school_name', 'flat_model', 'flat_type'
        ]
    }
}

# Export artifact bundle to your local hard drive
with open('../hdb_model_pipeline.pkl', 'wb') as f:
    pickle.dump(model_artifacts, f)

print("Pipeline artifacts successfully saved for deployment!")

Pipeline artifacts successfully saved for deployment!
